## 2026-06-09 WebDataScope Neutralization Version

- This copy is for field-aware neutralization experiments. `DATASET_ID`, `REGION`, `UNIVERSE`, and `DELAY` are independent parameters.
- WebDataScope field neutralization recommendation uses an efficiency score: keep neutralizations with Sharpe >= 0.35 and OS/IS count >= 20, then choose the highest `sharpe_ratio * log(osis_count + 1)`.
- If field-level WebDataScope data is missing, use dataset-level WebDataScope data with the same rule. If both are missing, assign by round-robin fallback: `MARKET`, `SECTOR`, `INDUSTRY`, `SUBINDUSTRY`.
- Simulation logs are separated by dataset, region, universe, delay, and neutralization.
- This notebook does not auto-submit alphas. Use the separate submission script only when explicitly needed.
- Submission scheduling controls are now configurable in this notebook: daily limit, 20-minute polling interval, and China-time blocked start/end window.
- Simulation pool sizing is now configurable by stage: first-order, second-order, and third-order each have separate slot and thread counts.
- Alpha fetch windows are now relative lookback windows, so track/submit candidate retrieval targets recent alphas and filters by `REGION` plus `UNIVERSE`.

## Base Update Notes

- Remote disconnect handling: simulation posting now catches host reset / request disconnect errors, waits 5 minutes for the first two reconnect attempts, then waits 1 hour for later reconnect attempts.
- Formal alpha submission: use `parse_renamed_alpha_log(...)` to extract `Renaming <alpha_id> to <name>` lines, then `submit_alpha_queue(...)` to submit one by one with a local daily limit of 4 successful submissions.
- Submission log: results are written to `runs/submissions/submission_log.jsonl`, so reruns skip already successful alpha ids and keep the daily quota count.
- Field-aware runs: save field-specific outputs under paths that include the data field id. WebDataScope neutralization stats can be used to choose a per-field neutralization before simulation.
- CLI shortcut: `python scripts/submit_alpha_queue.py --today 2026-06-09 --records "alpha_id=name,..."`.


## 1, Import Library

In [1]:
import os


In [2]:

from pathlib import Path
import importlib
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import consultant_core.machine_lib as machine_lib

# 修改 src/consultant_core/machine_lib.py 后，重新加载库代码。
machine_lib = importlib.reload(machine_lib)

check_submission = machine_lib.check_submission
first_order_factory = machine_lib.first_order_factory
get_alphas = machine_lib.get_alphas
get_alphas_full = machine_lib.get_alphas_full
get_daily_alpha_count = machine_lib.get_daily_alpha_count
get_recent_alphas = machine_lib.get_recent_alphas
get_datafields = machine_lib.get_datafields
get_group_second_order_factory = machine_lib.get_group_second_order_factory
load_task_pool = machine_lib.load_task_pool
load_webdatascope_info = machine_lib.load_webdatascope_info
login = machine_lib.login
multi_simulate = machine_lib.multi_simulate
next_simulation_start = machine_lib.next_simulation_start
parse_renamed_alpha_log = machine_lib.parse_renamed_alpha_log
process_datafields = machine_lib.process_datafields
prune = machine_lib.prune
daily_alpha_date_range = machine_lib.daily_alpha_date_range
recent_alpha_date_range = machine_lib.recent_alpha_date_range
recommended_neutralization_table = machine_lib.recommended_neutralization_table
split_processed_datafields_by_neutralization = machine_lib.split_processed_datafields_by_neutralization
submit_alpha_queue = machine_lib.submit_alpha_queue
submit_alpha_queue_until_limit = machine_lib.submit_alpha_queue_until_limit
trade_when_factory = machine_lib.trade_when_factory
ts_ops = machine_lib.ts_ops
view_alphas = machine_lib.view_alphas

# ===== 核心运行参数 =====
# 数据集 ID：在 WorldQuant Brain 的 Data 页面查看，例如 macro38。
DATASET_ID = "macro38"
# 国家/地区：用于数据字段、回测和 alpha 候选筛选，三处必须保持一致。
REGION = "USA"
# 股票池：用于数据字段、回测，并在拉取 alpha 后再次过滤。
UNIVERSE = "TOP3000"
# 数据延迟：用于读取数据字段，并写入日志目录名。
DELAY = 1

# ===== 回测任务池参数：load_task_pool(槽位数 * 线程数) =====
# 一阶槽位数：每个 multi-simulation 请求中包含的 alpha 数量。
FO_POOL_CHILDREN = 6
# 一阶线程数：每个 pool 中同时发送的 multi-simulation 请求数量。
FO_POOL_THREADS = 10
# 二阶槽位数。
SO_POOL_CHILDREN = 5
# 二阶线程数。
SO_POOL_THREADS = 8
# 三阶槽位数。
TH_POOL_CHILDREN = 5
# 三阶线程数。
TH_POOL_THREADS = 8
# ===== 最近 alpha 拉取参数 =====
# 日期时区：用于计算“最近 N 天”和“当天”的起止日期，默认按中国时间。
ALPHA_DATE_TIMEZONE = "Asia/Shanghai"
# 候选预拉取倍数：API 可直接按 REGION 查，但 universe 需要拉回后过滤。
# 例：目标需要 100 个 alpha，倍数为 3 时先拉 300 个，再按 UNIVERSE 过滤后取前 100 个。
ALPHA_FETCH_LIMIT_MULTIPLIER = 3
# 每日 alpha 数量上限：用于判断当天是否继续生成/回测 alpha，默认 4500。
# 对比方式：查询当天 dateCreated 的 alpha 数量，不限制 REGION，然后与该上限比较。
DAILY_ALPHA_LIMIT = 4500
# 每日数量统计最大拉取条数：通常等于 DAILY_ALPHA_LIMIT，避免为了计数拉取过多数据。
DAILY_ALPHA_COUNT_FETCH_LIMIT = DAILY_ALPHA_LIMIT
# 每日数量统计 usage：沿用 get_alphas/get_alphas_full 的 usage 语义。
# track 会按追踪逻辑统计；submit 会按提交候选逻辑统计。
DAILY_ALPHA_COUNT_USAGE = "track"
# 每日数量统计 status：默认与 get_alphas_full 当前查询一致，统计 UNSUBMITTED/IS_FAIL。
# 如果你确认要统计其它状态，再按 WorldQuant API 的 status 过滤语法调整。
DAILY_ALPHA_COUNT_STATUS = "UNSUBMITTED%1FIS_FAIL"
# 一阶追踪回看天数：从今天往前看多少天，用于获取最近的一阶可改进 alpha。
FO_TRACK_LOOKBACK_DAYS = 90
# 一阶追踪 Sharpe 阈值：低于该值的 alpha 不进入一阶追踪候选。
FO_TRACK_SHARPE = 1.0
# 一阶追踪 Fitness 阈值：低于该值的 alpha 不进入一阶追踪候选。
FO_TRACK_FITNESS = 0.7
# 一阶追踪最多保留数量：按查询排序和 UNIVERSE 过滤后最多保留多少个。
FO_TRACK_ALPHA_NUM = 100
# 二阶追踪回看天数：从今天往前看多少天，用于获取最近的二阶可改进 alpha。
SO_TRACK_LOOKBACK_DAYS = 90
# 二阶追踪 Sharpe 阈值：低于该值的 alpha 不进入二阶追踪候选。
SO_TRACK_SHARPE = 1.3
# 二阶追踪 Fitness 阈值：低于该值的 alpha 不进入二阶追踪候选。
SO_TRACK_FITNESS = 0.8
# 二阶追踪最多保留数量：按查询排序和 UNIVERSE 过滤后最多保留多少个。
SO_TRACK_ALPHA_NUM = 100
# 提交候选回看天数：从今天往前看多少天，用于获取最近可提交 alpha。
SUBMIT_LOOKBACK_DAYS = 30
# 提交候选 Sharpe 阈值：低于该值的 alpha 不进入提交候选。
SUBMIT_SHARPE = 1.58
# 提交候选 Fitness 阈值：低于该值的 alpha 不进入提交候选。
SUBMIT_FITNESS = 1.0
# 提交候选最多保留数量：按查询排序和 UNIVERSE 过滤后最多保留多少个。
SUBMIT_ALPHA_NUM = 200
# 每日提交上限：按中国日期统计，当天成功提交数量达到该值后停止提交。
SUBMISSION_DAILY_LIMIT = 4
# 轮询间隔：达到每日上限后，每隔多少分钟重新查询当天成功提交数量。
SUBMISSION_POLL_MINUTES = 20
# 禁投开始时间：中国时间。设置为 None 可关闭禁投时间段。
SUBMISSION_BLOCKED_START = "14:00"
# 禁投结束时间：中国时间。在 [开始时间, 结束时间) 内不会提交任何 alpha。
SUBMISSION_BLOCKED_END = "16:00"
# 提交控制时区：每日限额和禁投时间段都按这个时区判断。
SUBMISSION_TIMEZONE = "Asia/Shanghai"


def run_simulation_stage(pools, stage_name, log_filename, dataset_id=DATASET_ID, neut="SUBINDUSTRY", region=REGION, universe=UNIVERSE, delay=DELAY):
    log_path = PROJECT_ROOT / "runs" / "alpha_machine" / dataset_id / f"{region}_D{delay}_{universe}" / neut / log_filename
    start = next_simulation_start(log_path)
    print(f"Continue {stage_name} from pool {start}. log: {log_path}")
    multi_simulate(pools, neut, region, universe, start, log_path=log_path)
    return log_path


In [3]:
# 每日 alpha 数量检查：用于决定今天是否还继续跑回测生成新 alpha。
# 统计规则：只按当天 dateCreated 查询，REGION 不限制；如果数量 >= DAILY_ALPHA_LIMIT，建议暂停新的回测任务。
today_alpha_count = get_daily_alpha_count(
    alpha_num=DAILY_ALPHA_COUNT_FETCH_LIMIT,
    usage=DAILY_ALPHA_COUNT_USAGE,
    timezone_name=ALPHA_DATE_TIMEZONE,
    status=DAILY_ALPHA_COUNT_STATUS,
)
print(f"今日 alpha 数量: {today_alpha_count}/{DAILY_ALPHA_LIMIT}")
if today_alpha_count >= DAILY_ALPHA_LIMIT:
    print("今日 alpha 数量已达到上限，建议暂停新的回测任务。")
else:
    print("今日 alpha 数量未达到上限，可以继续回测。")


今日 alpha 数量: 0/4500
今日 alpha 数量未达到上限，可以继续回测。


## 2, 登录
<div style="margin-left: 20px;">
1, 在machine_lib文件的login方法中填写用户名和密码后保存，然后来到本文件Restart Kernal后重新import machine_lib后才在本文件生效
</div>

<div style="margin-left: 20px;">
2, 打印INVALID_CREDENTAIL即登录失败，打印自己的user_id信息才是登录成功。
</div>

In [4]:
s = login()


## WebDataScope Neutralization Workflow

This section reads WebDataScope neutralization statistics, recommends one neutralization per field, and splits simulations by neutralization.


In [5]:
# Load WebDataScope neutralization data from consultant/scripts/WebDataScope.
webdatascope_info = load_webdatascope_info(PROJECT_ROOT / "scripts" / "WebDataScope")

df = get_datafields(s, dataset_id=DATASET_ID, region=REGION, universe=UNIVERSE, delay=DELAY)
neutralization_recommendations = recommended_neutralization_table(
    df,
    webdatascope_info,
    dataset_id=DATASET_ID,
    region=REGION,
    universe=UNIVERSE,
    delay=DELAY,
)

recommendation_path = (
    PROJECT_ROOT / "runs" / "alpha_machine" / DATASET_ID / f"{REGION}_D{DELAY}_{UNIVERSE}" / "neutralization_recommendations.csv"
)
recommendation_path.parent.mkdir(parents=True, exist_ok=True)
neutralization_recommendations.to_csv(recommendation_path, index=False, encoding="utf-8-sig")
print(f"Saved recommendation table: {recommendation_path}")
neutralization_recommendations.head(20)


Saved recommendation table: d:\code\WorldQuant Brain\consultant\runs\alpha_machine\macro38\USA_D1_TOP3000\neutralization_recommendations.csv


,field_id,dataset_id,region,universe,delay,recommended_neutralization,recommendation_source,selection_rule,count,percentage,sharpe_ratio,osis_count,score
0,mcr38_adx_direction,macro38,USA,TOP3000,1,SUBINDUSTRY,webdatascope_datafield,efficiency_score,242,26.918799,0.673720,241,3.698006
1,mcr38_adx_signal,macro38,USA,TOP3000,1,SUBINDUSTRY,webdatascope_datafield,efficiency_score,387,31.591837,0.614967,385,3.662644
2,mcr38_adx_strength,macro38,USA,TOP3000,1,SUBINDUSTRY,webdatascope_datafield,efficiency_score,370,40.305011,0.680325,370,4.024938
3,mcr38_bb_direction,macro38,USA,TOP3000,1,STATISTICAL,webdatascope_datafield,efficiency_score,147,11.094340,1.024525,141,5.077368
4,mcr38_bb_signal,macro38,USA,TOP3000,1,SUBINDUSTRY,webdatascope_datafield,efficiency_score,314,34.966592,0.676158,312,3.885341
5,mcr38_bb_strength,macro38,USA,TOP3000,1,SUBINDUSTRY,webdatascope_datafield,efficiency_score,143,35.221675,0.701200,142,3.479944
6,mcr38_cci_40_direction,macro38,USA,TOP3000,1,STATISTICAL,webdatascope_datafield,efficiency_score,54,8.398134,1.098102,54,4.400460
7,mcr38_cci_40_signal,macro38,USA,TOP3000,1,SUBINDUSTRY,webdatascope_datafield,efficiency_score,130,27.196653,0.659918,130,3.217228
8,mcr38_cci_40_strength,macro38,USA,TOP3000,1,SUBINDUSTRY,webdatascope_datafield,efficiency_score,130,32.581454,0.644714,130,3.143108
9,mcr38_cci_60_direction,macro38,USA,TOP3000,1,STATISTICAL,webdatascope_datafield,efficiency_score,65,12.871287,1.030845,64,4.303146


In [6]:
# Build first-order simulation pools by recommended neutralization.
fields_by_neutralization = split_processed_datafields_by_neutralization(df, neutralization_recommendations)

fo_pools_by_neutralization = {}
for neut, field_exprs in fields_by_neutralization.items():
    first_order = first_order_factory(field_exprs, ts_ops)
    fo_alpha_list = [(alpha, 1) for alpha in first_order]
    fo_pools_by_neutralization[neut] = load_task_pool(fo_alpha_list, FO_POOL_CHILDREN, FO_POOL_THREADS)
    print(neut, "fields", len(field_exprs), "alphas", len(first_order), "pools", len(fo_pools_by_neutralization[neut]))

# Uncomment to run first-order simulations per neutralization.
# for neut, pools in fo_pools_by_neutralization.items():
#     run_simulation_stage(
#         pools,
#         f"first-order {neut}",
#         "fo_simulation_progress.jsonl",
#         dataset_id=DATASET_ID,
#         neut=neut,
#         region=REGION,
#         universe=UNIVERSE,
#         delay=DELAY,
#     )


SUBINDUSTRY fields 40 alphas 2240 pools 38
STATISTICAL fields 6 alphas 336 pools 6
FAST fields 1 alphas 56 pools 1
MARKET fields 2 alphas 112 pools 2
INDUSTRY fields 7 alphas 392 pools 7


## 3, 获取数据字段
<div style="margin-left: 20px;">
在官网Data页面中显示的为自己目前有权限的数据集，在数据集Description面板下可以看到dataset_id
</div>

In [7]:
df = get_datafields(s, dataset_id=DATASET_ID, region=REGION, universe=UNIVERSE, delay=DELAY)
df


,id,description,dataset,category,subcategory,region,delay,universe,type,dateCoverage,coverage,userCount,alphaCount,pyramidMultiplier,themes,dateCreated
0,mcr38_adx_direction,Indicates if the ADX signal is strengthening (...,"{'id': 'macro38', 'name': 'Technical Ratings M...","{'id': 'macro', 'name': 'Macro'}","{'id': 'macro-macroeconomic-activities', 'name...",USA,1,TOP3000,MATRIX,0.9507,1.0000,248,792,1.1,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase...",2022-10-01
1,mcr38_adx_signal,S(sell)/B(buy)/H(hold) for average directional...,"{'id': 'macro38', 'name': 'Technical Ratings M...","{'id': 'macro', 'name': 'Macro'}","{'id': 'macro-macroeconomic-activities', 'name...",USA,1,TOP3000,MATRIX,NaN,0.9796,298,979,1.1,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase...",2022-10-01
2,mcr38_adx_strength,Strength rating (1-5) of the Average Direction...,"{'id': 'macro38', 'name': 'Technical Ratings M...","{'id': 'macro', 'name': 'Macro'}","{'id': 'macro-macroeconomic-activities', 'name...",USA,1,TOP3000,MATRIX,0.9507,1.0000,218,847,1.1,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase...",2022-10-01
3,mcr38_bb_direction,Indicates if the Bollinger Band signal is stre...,"{'id': 'macro38', 'name': 'Technical Ratings M...","{'id': 'macro', 'name': 'Macro'}","{'id': 'macro-macroeconomic-activities', 'name...",USA,1,TOP3000,MATRIX,0.9507,1.0000,403,1486,1.1,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase...",2022-10-01
4,mcr38_bb_signal,S(sell)/B(buy)/H(hold) for bollinger band (BB)...,"{'id': 'macro38', 'name': 'Technical Ratings M...","{'id': 'macro', 'name': 'Macro'}","{'id': 'macro-macroeconomic-activities', 'name...",USA,1,TOP3000,MATRIX,NaN,0.9796,226,795,1.1,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase...",2022-10-01
5,mcr38_bb_strength,"Signal strength, from 1 to 5, for the Bollinge...","{'id': 'macro38', 'name': 'Technical Ratings M...","{'id': 'macro', 'name': 'Macro'}","{'id': 'macro-macroeconomic-activities', 'name...",USA,1,TOP3000,MATRIX,0.9507,1.0000,109,316,1.1,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase...",2022-10-01
6,mcr38_cci_40_direction,Direction and strength of the 40-period Commod...,"{'id': 'macro38', 'name': 'Technical Ratings M...","{'id': 'macro', 'name': 'Macro'}","{'id': 'macro-macroeconomic-activities', 'name...",USA,1,TOP3000,MATRIX,0.9507,1.0000,165,634,1.1,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase...",2022-10-01
7,mcr38_cci_40_signal,S(sell)/B(buy)/H(hold) for commodity channel i...,"{'id': 'macro38', 'name': 'Technical Ratings M...","{'id': 'macro', 'name': 'Macro'}","{'id': 'macro-macroeconomic-activities', 'name...",USA,1,TOP3000,MATRIX,NaN,0.9796,119,263,1.1,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase...",2022-10-01
8,mcr38_cci_40_strength,"Signal strength, from 1 to 5, for the Commodit...","{'id': 'macro38', 'name': 'Technical Ratings M...","{'id': 'macro', 'name': 'Macro'}","{'id': 'macro-macroeconomic-activities', 'name...",USA,1,TOP3000,MATRIX,0.9507,1.0000,111,373,1.1,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase...",2022-10-01
9,mcr38_cci_60_direction,Direction and strength of the 60-period Commod...,"{'id': 'macro38', 'name': 'Technical Ratings M...","{'id': 'macro', 'name': 'Macro'}","{'id': 'macro-macroeconomic-activities', 'name...",USA,1,TOP3000,MATRIX,0.9507,1.0000,129,477,1.1,"[{'id': 'Jypw8X4', 'name': 'USA/D1 Fast Datase...",2022-10-01


## 4，数据字段预处理

<div style="margin-left: 20px;">
1, matrix, vector 数据类型
</div>

<div style="margin-left: 20px;">
2, ts_backfill 回填缺失值，提高数据Coverage 
</div>

<div style="margin-left: 20px;">
2, winsorize 去极值
</div>

In [8]:
pc_fields = process_datafields(df)
len(pc_fields)


56

## 5, Alpha factory 
<div style="margin-left: 20px;">
在factory方法中将数据字段与操作符组装成alpha表达式
</div>

In [9]:
first_order = first_order_factory(pc_fields, ts_ops)
print(first_order[:10])
print(len(first_order))


['winsorize(ts_backfill(mcr38_adx_direction, 120), std=4)', 'ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 5)', 'ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 22)', 'ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 66)', 'ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 120)', 'ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 240)', 'ts_zscore(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 5)', 'ts_zscore(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 22)', 'ts_zscore(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 66)', 'ts_zscore(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 120)']
3136


## 6, 回测前载入

<div style="margin-left: 20px;">
1, alpha表达式与初始decay配对
</div>

<div style="margin-left: 20px;">
2, 保持固定顺序，便于失败后按日志续跑
</div>

<div style="margin-left: 20px;">
2, Load task pool数据结构
</div>

In [10]:
# 赋予alpha表达式一个初始decay
init_decay = 6
fo_alpha_list = []

for alpha in first_order:
    fo_alpha_list.append((alpha, init_decay))

print("数量: %s"%len(fo_alpha_list))
print(fo_alpha_list[:5])


数量: 3136
[('winsorize(ts_backfill(mcr38_adx_direction, 120), std=4)', 6), ('ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 5)', 6), ('ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 22)', 6), ('ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 66)', 6), ('ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 120)', 6)]


In [11]:
# Load alphas to task pools
fo_pools = load_task_pool(fo_alpha_list, FO_POOL_CHILDREN, FO_POOL_THREADS)
print(fo_pools[0])


[[('winsorize(ts_backfill(mcr38_adx_direction, 120), std=4)', 6), ('ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 5)', 6), ('ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 22)', 6), ('ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 66)', 6), ('ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 120)', 6), ('ts_rank(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 240)', 6)], [('ts_zscore(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 5)', 6), ('ts_zscore(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 22)', 6), ('ts_zscore(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 66)', 6), ('ts_zscore(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 120)', 6), ('ts_zscore(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 240)', 6), ('ts_delta(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 5)', 6)], [('ts_delta(winsorize(ts_backfill(mcr38_adx_direction, 120), std=4), 22)

## 7, 回测

In [ ]:
# Simulate First Order
fo_log = run_simulation_stage(fo_pools, "一阶", "fo_simulation_progress.jsonl")


Continue 一阶 from pool 0. log: d:\code\WorldQuant Brain\consultant\runs\alpha_machine\macro38\USA_D1_TOP3000\SUBINDUSTRY\fo_simulation_progress.jsonl
location key error: {"detail":"CONCURRENT_SIMULATION_LIMIT_EXCEEDED"}


## 8, 筛选Alpha


<div style="margin-left: 20px;">
1, get_alpha：截取有潜力提升表现至可以提交的alpha进入下一阶
</div>

<div style="margin-left: 20px;">
2, 剪枝Prune：精减相似alpha，提高回测资源利用率
</div>

In [ ]:
## get promising alphas to improve in the next order
fo_tracker = get_recent_alphas(FO_TRACK_LOOKBACK_DAYS, FO_TRACK_SHARPE, FO_TRACK_FITNESS, REGION, UNIVERSE, FO_TRACK_ALPHA_NUM, "track", timezone_name=ALPHA_DATE_TIMEZONE, fetch_limit_multiplier=ALPHA_FETCH_LIMIT_MULTIPLIER)
print(len(fo_tracker))


#### Prune 剪枝

In [ ]:
fo_layer = prune(fo_tracker, 'anl4', 5)

# 剪枝后数量
print(len(fo_layer))


## 9, 二阶提升
### ts_ops(field, days) -> group_ops(ts_ops(field, days), group)

In [ ]:
so_alpha_list = []
group_ops = ["group_neutralize", "group_rank", "group_zscore"]

for expr, decay in fo_layer:
    for alpha in get_group_second_order_factory([expr], group_ops, "USA"):
        so_alpha_list.append((alpha,decay))

print(len(so_alpha_list))
print(so_alpha_list[:3])


### Simulate second order

In [ ]:
so_pools = load_task_pool(so_alpha_list, SO_POOL_CHILDREN, SO_POOL_THREADS)
so_log = run_simulation_stage(so_pools, "二阶", "so_simulation_progress.jsonl")


## 10，三阶提升
group_ops(ts_ops(field, days), group) -> trade_when(entre_event, group_ops(ts_ops(field, days), group), exit_event)

In [ ]:
## get promising alphas from second order to improve in the third order
so_tracker = get_recent_alphas(SO_TRACK_LOOKBACK_DAYS, SO_TRACK_SHARPE, SO_TRACK_FITNESS, REGION, UNIVERSE, SO_TRACK_ALPHA_NUM, "track", timezone_name=ALPHA_DATE_TIMEZONE, fetch_limit_multiplier=ALPHA_FETCH_LIMIT_MULTIPLIER)

so_layer = prune(so_tracker, 'anl4', 5)
th_alpha_list = []

for expr, decay in so_layer:
    for alpha in trade_when_factory("trade_when",expr,"USA"):
        th_alpha_list.append((alpha,decay))

print("三阶表达式数量:%s"%len(th_alpha_list))


### Simulate Third Order

In [ ]:
# Simulate third order
th_pools = load_task_pool(th_alpha_list, TH_POOL_CHILDREN, TH_POOL_THREADS)
th_log = run_simulation_stage(th_pools, "三阶", "th_simulation_progress.jsonl")


## 11, 获取可提交的Alpha

<div style="margin-left: 20px;">
1, 拉取sharpe,fitness达到提交要求的alpha
</div>

<div style="margin-left: 20px;">
2, Check Submission：检查其他Test是否达到要求
</div>

<div style="margin-left: 20px;">
2, view_alphas 对可以提交的alpha进行排序
</div>


In [ ]:
# 1.58 sharpe, 1 fitness, "submit"参数
th_tracker = get_recent_alphas(SUBMIT_LOOKBACK_DAYS, SUBMIT_SHARPE, SUBMIT_FITNESS, REGION, UNIVERSE, SUBMIT_ALPHA_NUM, "submit", timezone_name=ALPHA_DATE_TIMEZONE, fetch_limit_multiplier=ALPHA_FETCH_LIMIT_MULTIPLIER)


In [ ]:
## 将get的alpha的id取出至stone_bag，用api check submission
stone_bag = []
for alpha in th_tracker:
    stone_bag.append(alpha[0])
print(len(stone_bag))
gold_bag = []
check_submission(stone_bag, gold_bag, 0)


In [ ]:
# 打印可提交的alpha信息并按sharpe排序，在网页上找到alpha手动提交
view_alphas(gold_bag)


In [ ]:

# 定时正式提交示例。
# 这里保留为手动步骤：只有填入 submit_records，并把 dry_run 改成 False 后才会真实提交。
rename_log_text = """
# 粘贴类似这样的日志行：2026-06-08 20:05:23,561 - INFO - Renaming omYz2096 to RA_0.55
"""
submit_records = parse_renamed_alpha_log(rename_log_text)

# 安全默认值：dry_run=True。确认要真实提交时，再改成 False。
# scheduled_submission_summary = submit_alpha_queue_until_limit(
#     submit_records,
#     daily_limit=SUBMISSION_DAILY_LIMIT,
#     poll_seconds=SUBMISSION_POLL_MINUTES * 60,
#     blocked_start=SUBMISSION_BLOCKED_START,
#     blocked_end=SUBMISSION_BLOCKED_END,
#     timezone_name=SUBMISSION_TIMEZONE,
#     dry_run=True,
# )
# scheduled_submission_summary


## 12, 微调可以提交的alpha

<div style="margin-left: 20px;">
1, 得到更好的表现
</div>
<div style="margin-left: 40px;">
调整中性化，操作符参数，Decay
</div>

<div style="margin-left: 20px;">
2, Alpha质量评估
</div>

<div style="margin-left: 40px;">
performance comparison，turnover，margin
</div>

<div style="margin-left: 20px;">
3, 鲁棒性评估，防止过拟合
</div>

<div style="margin-left: 40px;">
更改中性化，Rank，Binary Test...
</div>

### Appendix

In [ ]:
# 模板构建Factory实例

def template_factory(sent_fields, option_fields):
    alpha_list = []
    for sent_field in sent_fields:
        for opt_field in option_fields:
            alpha_list.append("log(1+sigmoid(ts_zscore(%s,30))*sigmoid(ts_zscore(%s,30))"%(sent_field, opt_field))
    return alpha_list 

opt_df = get_datafields(s, dataset_id = 'option8', region='USA', universe='TOP3000', delay=1)
opt_fields = opt_df[opt_df['type'] == "MATRIX"]["id"].tolist()
print(opt_fields)

sent_df = get_datafields(s, dataset_id = 'sentiment1', region='USA', universe='TOP3000', delay=1)
sent_fields = sent_df[sent_df['type'] == "MATRIX"]["id"].tolist()
print(sent_fields)

alpha_list = template_factory(sent_fields, opt_fields)
print(alpha_list)
